# DressMe · Objectif 1 — version Colab + Google Drive

Cette version garde **tout le projet dans Google Drive** :

```text
MyDrive/
└── dressme_obj1/
    ├── contracts/
    ├── evaluation/
    ├── recognition/
    ├── notebooks/
    ├── tests/
    └── data/
        ├── database/
        │   └── dressme_training_ready.csv
        ├── tunisian/
        │   └── images/
        ├── tunisian_dataset/
        ├── agent_final/
        ├── store/
        ├── training_json_results/
        ├── real30/
        └── inbox/
```

Le catalogue Fashion Product Images reste téléchargé automatiquement par `recognition/knowledge_base.py`.

Exécuter les cellules de haut en bas dans Google Colab.


In [ ]:
# 0. Installation Colab
!pip -q install -U transformers accelerate bitsandbytes kagglehub tqdm matplotlib pydantic scikit-learn pandas numpy pillow rembg onnxruntime


In [ ]:
# 1. Google Drive + chemins du projet

from google.colab import drive
from pathlib import Path
import os
import sys

# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

DRIVE_MOUNT = "/content/drive"

if not Path(DRIVE_MOUNT, "MyDrive").exists():
    drive.mount(DRIVE_MOUNT)


# ============================================================
# PROJECT ROOT
#
# Tout le projet est dans :
# MyDrive/dressme_obj1/
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/dressme_obj1"
)

DATA_ROOT = PROJECT_ROOT / "data"


# ============================================================
# DATA PATHS
# ============================================================

DATABASE_DIR = DATA_ROOT / "database"

TRAINING_READY = (
    DATABASE_DIR
    / "dressme_training_ready.csv"
)

MASTER_DATASET = (
    DATABASE_DIR
    / "dressme_master_dataset.csv"
)

TN_SOURCE_DIR = (
    DATA_ROOT
    / "tunisian"
)

TN_SOURCE_IMAGES = (
    TN_SOURCE_DIR
    / "images"
)

TN_DATASET_DIR = (
    DATA_ROOT
    / "tunisian_dataset"
)

TN_DATASET_IMAGES = (
    TN_DATASET_DIR
    / "images"
)

TN_LABELS = (
    TN_DATASET_DIR
    / "labels.csv"
)

STORE = (
    DATA_ROOT
    / "store"
)

AGENT_FINAL = (
    DATA_ROOT
    / "agent_final"
)

JSON_OUTPUT = (
    DATA_ROOT
    / "training_json_results"
)

INBOX = (
    DATA_ROOT
    / "inbox"
)

REAL30 = (
    DATA_ROOT
    / "real30"
)


# ============================================================
# CREATE OUTPUT FOLDERS IF MISSING
# ============================================================

for folder in [
    DATABASE_DIR,
    TN_SOURCE_IMAGES,
    TN_DATASET_IMAGES,
    STORE,
    AGENT_FINAL,
    JSON_OUTPUT,
    INBOX,
    REAL30,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# CHECK PROJECT CODE
# ============================================================

required_code = [
    PROJECT_ROOT / "recognition" / "service.py",
    PROJECT_ROOT / "recognition" / "config.py",
    PROJECT_ROOT / "recognition" / "tunisian.py",
    PROJECT_ROOT / "recognition" / "finetune.py",
    PROJECT_ROOT / "contracts" / "__init__.py",
    PROJECT_ROOT / "evaluation" / "__init__.py",
]

missing_code = [
    str(p)
    for p in required_code
    if not p.exists()
]

if missing_code:
    print("❌ Fichiers du projet manquants :")
    for p in missing_code:
        print(" -", p)

    raise RuntimeError(
        "Le dossier MyDrive/dressme_obj1/ doit contenir "
        "recognition/, contracts/ et evaluation/."
    )


# ============================================================
# CONFIGURE PYTHON IMPORT PATH
# ============================================================

os.environ["DRESSME_ROOT"] = str(DATA_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


# ============================================================
# LOAD DRESSME CONFIG
# ============================================================

from recognition.config import CFG
from contracts import load_type_dictionary

CFG.root = DATA_ROOT

# Keep Tunisian references in kNN/RAG
CFG.tn_refs = True

# Start without Tunisian-specific prompt.
# You can compare True/False later on DEV.
CFG.tunisian_prompt = False


# ============================================================
# FINAL CHECK
# ============================================================

print("=" * 80)
print("DRESSME COLAB + DRIVE CONFIGURATION")
print("=" * 80)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)

print(
    "\nTraining ready :",
    TRAINING_READY.exists(),
    TRAINING_READY
)

print(
    "Tunisian images:",
    TN_SOURCE_IMAGES.exists(),
    TN_SOURCE_IMAGES
)

print(
    "Tunisian labels:",
    TN_LABELS.exists(),
    TN_LABELS
)

print(
    "\nOfficial dictionary:",
    load_type_dictionary()
)

print("=" * 80)


if not TRAINING_READY.exists():
    raise FileNotFoundError(
        f"\nMissing:\n{TRAINING_READY}\n\n"
        "Put dressme_training_ready.csv inside "
        "MyDrive/dressme_obj1/data/database/"
    )


In [ ]:
# 2. Préparer automatiquement tunisian_dataset/labels.csv
# Source : data/database/dressme_training_ready.csv
# On garde UNIQUEMENT les lignes tunisiennes acceptées.
#
# Cette cellule ne duplique pas le dataset général :
# recognition/knowledge_base.py le charge déjà depuis Kaggle.

import shutil
import pandas as pd

if not TRAINING_READY.exists():
    raise FileNotFoundError(
        f"Fichier absent : {TRAINING_READY}\n"
        "Copie dressme_training_ready.csv dans data/database/."
    )

ready = pd.read_csv(TRAINING_READY, low_memory=False)

if "is_tunisian" not in ready.columns:
    raise RuntimeError(
        "La colonne 'is_tunisian' manque dans dressme_training_ready.csv."
    )

tn = ready[
    ready["is_tunisian"].fillna(0).astype(int).eq(1)
].copy()

if "usable_for_training" in tn.columns:
    tn = tn[
        tn["usable_for_training"].fillna(0).astype(int).eq(1)
    ].copy()

if "needs_review" in tn.columns:
    tn = tn[
        tn["needs_review"].fillna(1).astype(int).eq(0)
    ].copy()

print("Lignes tunisiennes acceptées :", len(tn))

VALID_CATEGORIES = {
    "tshirt", "shirt", "top", "sweater",
    "jacket", "blazer", "coat", "dress",
    "skirt", "jeans", "pants", "shorts",
    "underwear", "shoes", "bag", "accessory",
    "traditional_garment",
}

VALID_COLORS = {
    "beige", "black", "blue", "brown", "green", "grey",
    "metallic", "multicolor", "orange", "pink", "purple",
    "red", "white", "yellow",
}

VALID_GENDERS = {"male", "female", "unisex"}
VALID_SEASONS = {"summer", "winter", "spring", "fall", "all_season"}
VALID_OCCASIONS = {"casual", "formal", "sport", "party", "traditional", "home"}

def clean(v):
    if pd.isna(v):
        return ""
    return str(v).strip().lower()

def norm_category(v):
    v = clean(v)
    aliases = {
        "t-shirt": "tshirt",
        "t_shirt": "tshirt",
        "tee": "tshirt",
        "trousers": "pants",
        "pant": "pants",
        "hoodie": "sweater",
        "sweatshirt": "sweater",
        "blouse": "top",
        "traditional": "traditional_garment",
        "traditional garment": "traditional_garment",
    }
    return aliases.get(v, v)

def norm_color(v):
    v = clean(v).replace(" ", "_")
    aliases = {
        "gray": "grey",
        "silver": "metallic",
        "gold": "metallic",
        "multi": "multicolor",
        "multi_color": "multicolor",
        "multicolored": "multicolor",
        "navy": "blue",
        "navy_blue": "blue",
        "cream": "beige",
        "unknown": "",
    }
    v = aliases.get(v, v)
    return v if v in VALID_COLORS else ""

def norm_gender(v):
    v = clean(v)
    aliases = {
        "men": "male",
        "man": "male",
        "women": "female",
        "woman": "female",
        "unknown": "",
    }
    v = aliases.get(v, v)
    return v if v in VALID_GENDERS else ""

def norm_season(v):
    v = clean(v)
    aliases = {
        "autumn": "fall",
        "all": "all_season",
        "all_seasons": "all_season",
        "unknown": "",
    }
    v = aliases.get(v, v)
    return v if v in VALID_SEASONS else ""

def norm_occasion(v):
    v = clean(v).replace(" ", "_")
    aliases = {
        "sports": "sport",
        "smart_casual": "casual",
        "everyday": "casual",
        "ethnic": "traditional",
        "ceremonial": "traditional",
        "ceremony": "traditional",
        "unknown": "",
    }
    v = aliases.get(v, v)
    return v if v in VALID_OCCASIONS else ""

def infer_subtype(row, category):
    if category != "traditional_garment":
        return ""
    garment = clean(row.get("garment_name", ""))
    for s in ("jebba", "farmla", "blouza", "kaftan", "fouta"):
        if s in garment:
            return s
    return "other"

def find_source_image(row):
    record_id = str(row.get("record_id", "")).strip()

    # 1. Chemin déjà présent dans le CSV, si encore valide
    old = row.get("image_path", "")
    if isinstance(old, str) and old.strip():
        p = Path(old)
        if p.is_file():
            return p

    # 2. data/tunisian/images/<record_id>.<ext>
    for ext in (".jpg", ".jpeg", ".png", ".webp"):
        p = TN_SOURCE_IMAGES / f"{record_id}{ext}"
        if p.is_file():
            return p

    return None

rows = []
missing = []
invalid = []

for _, r in tn.iterrows():
    record_id = str(r.get("record_id", "")).strip()
    category = norm_category(r.get("category", ""))

    if category not in VALID_CATEGORIES:
        invalid.append((record_id, category))
        continue

    src = find_source_image(r)

    if src is None:
        missing.append(record_id)
        continue

    dst = TN_DATASET_IMAGES / f"{record_id}{src.suffix.lower()}"

    if not dst.exists():
        shutil.copy2(src, dst)

    rows.append({
        "image_path": (Path("images") / dst.name).as_posix(),
        "category": category,
        "subtype": infer_subtype(r, category),
        "colour": norm_color(r.get("color", "")),
        "gender": norm_gender(r.get("gender", "")),
        "season": norm_season(r.get("season", "")),
        "occasion": norm_occasion(r.get("occasion", "")),
        "event": "",
        "origin": "tunisia_auto_qwen",
        "group_id": f"tn_{record_id}",
        "split": "",      # attribution automatique 60/20/20 par recognition/tunisian.py
        "cropped": "0",
    })

labels = pd.DataFrame(rows)
labels.to_csv(TN_LABELS, index=False, encoding="utf-8-sig")

print("labels.csv créé :", TN_LABELS)
print("Lignes valides  :", len(labels))
print("Images absentes :", len(missing), missing[:10])
print("Catégories invalides :", len(invalid), invalid[:10])

display(labels.head(20))


In [ ]:
# 3. Contrôle des données tunisiennes

from recognition.tunisian import report, load_tunisian

tab = report(CFG)

tn_all, issues = load_tunisian(CFG)

print("\nTotal lignes tunisiennes valides :", len(tn_all))

if len(tn_all):
    print("\nSplits :")
    print(tn_all["split"].value_counts())
    print("\nCatégories :")
    print(tn_all["category"].value_counts())


In [ ]:
# 4. Chargement de l'agent
# Qwen + FashionCLIP + base kNN catalogue + références tunisiennes train

from recognition.service import RecognitionService

svc = RecognitionService.from_config(CFG)

print(
    "✅ prêt | base catalogue :",
    len(svc.kb_df),
    "| références tunisiennes :",
    svc.n_tn_refs,
    "| test catalogue :",
    len(svc.test_df),
)


In [ ]:
# 5. Fine-tuning FashionCLIP sur le mix :
# catalogue Fashion Product Images + tunisian split=train
#
# Le catalogue général est chargé automatiquement par le projet.
# Aucun URL Kaggle à modifier.

RUN_FINETUNE = True

FINETUNED_ACTIVE = False

if RUN_FINETUNE:
    FINETUNED_ACTIVE = svc.enable_finetuned()

print("FashionCLIP fine-tuné actif :", FINETUNED_ACTIVE)


In [ ]:
# 6. Tester UNE image locale
#
# Mets une image dans :
# data/inbox/
#
# La cellule prend automatiquement la première image trouvée.

from pathlib import Path

images = sorted([
    p for p in INBOX.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
])

if not images:
    print("Ajoute d'abord une image dans :", INBOX)
else:
    test_image = images[0]
    print("Image test :", test_image)

    result = svc.recognize(
        test_image,
        item_id="local_test_0001",
        out_dir=STORE,
        mode="full",
    )

    print("status   :", result.status)
    print("category :", result.category)
    print("reasons  :", result.reasons)

    if result.card:
        print(result.card.model_dump_json(indent=2))


In [ ]:
# 7. Évaluation catalogue
# CORRECTION : le dossier du projet s'appelle evaluation/, pas eval/

from evaluation.eval_catalogue import run_catalogue_eval

run_catalogue_eval(svc, "v1")
run_catalogue_eval(svc, "llm_only", "llm_only")
run_catalogue_eval(svc, "knn_only", "knn_only")


In [ ]:
# 8. Évaluation real30
#
# Attendu :
# data/real30/labels.csv
#
# avec les colonnes prévues par evaluation/eval_real.py

from evaluation.eval_real import run_real_eval

REAL30_LABELS = REAL30 / "labels.csv"

if REAL30_LABELS.exists():
    out_real = run_real_eval(
        svc,
        REAL30_LABELS,
    )
else:
    print("Pas encore de fichier :", REAL30_LABELS)
    print("Cette étape est optionnelle tant que real30/labels.csv n'est pas prêt.")


In [ ]:
# 9. Export d'un dossier d'images utilisateur
#
# Mets tes images/crops dans data/inbox/.
# Le projet écrit automatiquement :
#
# data/store/items/*.json
# data/store/items/*_crop.png
# data/store/emb/*.npy
# data/store/wardrobe.json
# data/store/recognition_report.json

from recognition.batch import recognize_folder

input_images = [
    p for p in INBOX.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
]

if input_images:
    summary = recognize_folder(
        svc,
        INBOX,
        STORE,
        crop=True,
    )
    print(summary)
else:
    print("Aucune image dans :", INBOX)


In [ ]:
# 10. OPTIONNEL : un JSON par image du split TRAIN tunisien
#
# Cette cellule ne refait pas les milliers d'images génériques du catalogue.
# Elle exporte le split tunisien réellement utilisé comme références/fine-tuning.
#
# Pour tester d'abord :
# EXPORT_LIMIT = 5
#
# Pour tout exporter :
# EXPORT_LIMIT = None

import json
from recognition.tunisian import load_tunisian, prepare_crops

EXPORT_LIMIT = 5

train_df, issues = load_tunisian(CFG, "train")

if len(train_df) == 0:
    print("Aucune image tunisienne dans le split train.")
else:
    train_df = prepare_crops(
        CFG,
        train_df,
        svc.cropper,
    )

    if EXPORT_LIMIT is not None:
        train_df = train_df.head(EXPORT_LIMIT)

    run_name = "finetuned" if FINETUNED_ACTIVE else "frozen"
    run_dir = JSON_OUTPUT / run_name
    per_image_dir = run_dir / "per_image"
    result_store = run_dir / "runtime_store"

    per_image_dir.mkdir(parents=True, exist_ok=True)
    result_store.mkdir(parents=True, exist_ok=True)

    all_results = []

    for row in tqdm(
        train_df.itertuples(index=False),
        total=len(train_df),
        desc="Tunisian TRAIN -> JSON",
    ):
        item_id = Path(row.image_path).stem
        json_path = per_image_dir / f"{item_id}.json"

        if json_path.exists():
            payload = json.loads(json_path.read_text(encoding="utf-8"))
            all_results.append(payload)
            continue

        try:
            result = svc.recognize(
                row.crop_path,
                item_id=item_id,
                out_dir=result_store,
                mode="full",
            )

            payload = {
                "id": item_id,
                "input": {
                    "image_path": row.abs_path,
                    "crop_path": row.crop_path,
                    "dataset_label": {
                        "category": row.category,
                        "subtype": row.subtype,
                        "colour": row.colour,
                        "gender": row.gender,
                        "season": row.season,
                        "occasion": row.occasion,
                    },
                },
                "output": result.model_dump(mode="json"),
                "pipeline": {
                    "fashionclip_finetuned": bool(FINETUNED_ACTIVE),
                    "qwen_model": CFG.llm_id,
                    "fashionclip_model": CFG.embedder_id,
                },
            }

        except Exception as e:
            payload = {
                "id": item_id,
                "input": {
                    "image_path": row.abs_path,
                    "crop_path": row.crop_path,
                },
                "output": {
                    "status": "error",
                    "category": "unknown",
                    "error": repr(e),
                },
            }

        json_path.write_text(
            json.dumps(payload, indent=2, ensure_ascii=False, default=str),
            encoding="utf-8",
        )

        all_results.append(payload)

    global_json = run_dir / "all_training_results.json"

    global_json.write_text(
        json.dumps(all_results, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

    print("✅ JSON par image :", per_image_dir)
    print("✅ JSON global    :", global_json)
    print("Nombre exporté    :", len(all_results))


In [ ]:
# 11. Mock + schéma JSON

from recognition.mock import MockRecognizer
from contracts import export_json_schema

print(
    MockRecognizer()
    .recognize("x.png")
    .card
    .model_dump_json(indent=2)[:400]
)

schema_path = (
    PROJECT_ROOT
    / "contracts"
    / "schema"
    / "itemcard_v1.schema.json"
)

export_json_schema(schema_path)

print("Schéma écrit :", schema_path)
